In [3]:
from src.builders.SQL.SQLCaller import SQL_ETL
from typing import Iterable, Iterator, TypeVar
from itertools import islice
from src.builders.Neo4j.schema.nodes import build_neo4j_entity
from itertools import groupby



T = TypeVar("T")

def batched(iterable: Iterable[T], n: int) -> Iterator[list[T]]:
    it = iter(iterable)
    while chunk := list(islice(it, n)):
        yield chunk

UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'
sql_caller = SQL_ETL(run_id=UUID)

all_entities = sql_caller.sql_state.fetch_diff_entities(table_name='entities')

for batch in all_entities:
    for chunk in batched(batch, 100):
        entity_nodes = [build_neo4j_entity(record["entity_type"], record) for record in chunk]
        print(entity_nodes)
    break

TypeError: StructureNode must define __label__

In [2]:
from src.models.kegg import EntityType

EntityType("ortholog")

<EntityType.ORTHOLOG: 'ortholog'>

In [1]:
#1. Creation test logic
from src.builders.SQL.schema.definitions import table_schemas
from src.builders.SQL.schema.table_registry import table_managers
from src.builders.SQL.schema.table_utils import resolve_staging_plan

for table_name, schema in table_schemas.items():

    table_manager = table_managers[table_name]
    print(table_name, table_manager.strategy)
    # print("Table Creation:")
    # print(table_manager.strategy._build_staging_create(table_manager.schema, f"staging.{table_name}"))
    # print(table_manager.strategy._build_production_create(table_manager.schema, f"dbo.{table_name}"))

    # print("Staging Plan:")
    # stage_sql = table_manager.strategy.stage_data(table_manager.schema)
    # print(stage_sql.sql, stage_sql.num_params)
    # print(table_manager.strategy.upsert_data(table_manager.schema, run_id='any'))

    print("Resolve staging Plan")
    for column_plan in resolve_staging_plan(table_manager.schema):
        print(column_plan)

    print(table_manager.strategy._build_staging_create(table_manager.schema, f"staging.{table_name}"))
    print(table_manager.strategy._build_production_create(table_manager.schema, f"dbo.{table_name}"))
    print()

    print()

    print(table_manager.strategy.stage_data(table_manager.schema).sql)

    for upsert in table_manager.strategy.upsert_data(table_manager.schema, run_id='anyrunID'):
        print(upsert)

PathwayIds <src.builders.SQL.schema.strategies.StageUpsertStrategies.SQLStageUpsertDiff object at 0x7fac1a379610>
Resolve staging Plan
ColumnPlan(production_col='pathway_id', staging_col='pathway_id', staging_type='VARCHAR(20)', deffered_fk=None, reused_column=False)
ColumnPlan(production_col='name', staging_col='name', staging_type='VARCHAR(MAX)', deffered_fk=None, reused_column=False)

        IF NOT EXISTS (
            SELECT 1 FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'PathwayIds'
        )
        BEGIN
            CREATE TABLE staging.PathwayIds (
   pathway_id VARCHAR(20),
     name VARCHAR(MAX)
)
        END
        

        IF NOT EXISTS (
            SELECT 1 FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'dbo' AND TABLE_NAME = 'PathwayIds'
        )
        BEGIN
            CREATE TABLE dbo.PathwayIds (
   pathway_id VARCHAR(20) PRIMARY KEY,
     name VARCHAR(MAX)
)
        END
        



        INSERT IN